In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_inventory = spark.read.parquet("abfss://RETAIL_DE@onelake.dfs.fabric.microsoft.com/RETAIL_LH.Lakehouse/Files/Bronze/inventory_data.parquet")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 4, Finished, Available, Finished, False)

In [3]:
display(df_inventory)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, feb2dcd8-7e3c-47c9-a01b-db2b048dd4fe)

In [58]:
df_orders = spark.read.parquet("abfss://RETAIL_DE@onelake.dfs.fabric.microsoft.com/RETAIL_LH.Lakehouse/Files/Bronze/orders_data.parquet")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 60, Finished, Available, Finished, False)

In [5]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d2daf4cc-06db-40ed-b35e-7c3c71eb8315)

In [6]:
df_returns = spark.read.parquet("abfss://RETAIL_DE@onelake.dfs.fabric.microsoft.com/RETAIL_LH.Lakehouse/Files/Bronze/returns_data.xlsx.parquet")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 8, Finished, Available, Finished, False)

In [7]:
display(df_returns)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50c43d0b-4e0d-452d-b4a2-4489c27af10a)

****Cleaning Orders Data****

In [8]:
from pyspark.sql.functions import *

df_orders = (
    df_orders
    # Fix inconsistent column names
    .withColumnRenamed("Order iD", "Order_ID")
    .withColumnRenamed("order_date", "Order_Date")
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 10, Finished, Available, Finished, False)

In [9]:
# Convert Order_Date to proper format, replacing "/" with "-" and casting to date
from pyspark.sql.functions import col, trim, regexp_replace, to_date, when, concat_ws, split

df_orders = df_orders.withColumn(
    "Order_Date",
    trim(col("Order_Date").cast("string"))
)

# Remove st, nd, rd, th
df_orders = df_orders.withColumn(
    "Order_Date",
    regexp_replace(
        col("Order_Date"),
        r"(\d+)(st|nd|rd|th)",
        "$1"
    )
)

# 2023/13/07 -> 2023-07-13
df_orders = df_orders.withColumn(
    "Order_Date",
    when(
        col("Order_Date").rlike(r"^\d{4}/\d{2}/\d{2}$"),
        concat_ws(
            "-",
            split(col("Order_Date"), "/").getItem(0),
            split(col("Order_Date"), "/").getItem(2),
            split(col("Order_Date"), "/").getItem(1)
        )
    ).otherwise(col("Order_Date"))
)

# Replace . and / with -
df_orders = df_orders.withColumn(
    "Order_Date",
    regexp_replace(col("Order_Date"), r"[/.]", "-")
)

# Convert remaining formats
df_orders = df_orders.withColumn(
    "Order_Date",
    when(
        col("Order_Date").rlike(r"^\d{4}-\d{2}-\d{2}$"),
        to_date(col("Order_Date"), "yyyy-MM-dd")
    )
    .when(
        col("Order_Date").rlike(r"^\d{2}-\d{2}-\d{4}$") &
        (col("Order_Date").substr(4, 2).cast("int") > 12),
        to_date(col("Order_Date"), "MM-dd-yyyy")
    )
    .when(
        col("Order_Date").rlike(r"^\d{2}-\d{2}-\d{4}$"),
        to_date(col("Order_Date"), "dd-MM-yyyy")
    )
    .when(
        col("Order_Date").rlike(r"^[A-Za-z]+ \d{1,2} \d{4}$"),
        to_date(col("Order_Date"), "MMMM d yyyy")
    )
)

df_orders.show(50, truncate=False)
   

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 11, Finished, Available, Finished, False)

+--------+-------+-------------------+-----+----------+-------------+---------------+------------+------------------------+---------------------+----------+--------------+
|Order_ID|cust_id|Product_Name       |Qty  |Order_Date|Order_Amount$|Delivery_Status|Payment_Mode|Ship_Address            |Email                |Promo_Code|Feedback_Score|
+--------+-------+-------------------+-----+----------+-------------+---------------+------------+------------------------+---------------------+----------+--------------+
|1001    |C001   |Apple iPhone13     |one  |2023-12-06|$799.0       |delivered      |CreditCard  |221B, baker st. , london|john.doe@@gmail..com |NEW2023   |5             |
|1002    |C_002  |samsung Galaxy s21 |NULL |2023-07-12|$1200        |Delivered      |PayPal      |Noida #Sector21         |NULL                 |Summer2023|NULL          |
|1003    |C003   |NULL               |3    |2023-07-15|NULL         |Pending        |DEBIT card  |123, main street        |jane_doe#email.co

In [10]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c3f0d07-fc4a-4e4b-bd86-26be81c58fa4)

In [11]:
# Standardize Customer ID format – uppercase and trimmed
from pyspark.sql.functions import col, trim, upper
df_orders = (
    df_orders
    .withColumn("CustomerID", trim(upper(col("cust_id"))))
    .drop("cust_id")
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 13, Finished, Available, Finished, False)

In [12]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3876753-8b0e-4545-a124-827cbced15af)

In [13]:

  # Clean emails – lowercase and trimmed
from pyspark.sql import functions as F

df_orders = (
    df_orders
    .withColumn(
        "Email",
        F.lower(F.trim(F.col("Email")))
    )
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 15, Finished, Available, Finished, False)

In [101]:
display(df_orders)

StatementMeta(, 7ab9786a-4584-42aa-995e-4500b3ae39de, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d8fab4e5-a1d3-4309-9259-80017517c08e)

In [14]:
from pyspark.sql import functions as F

df_orders = (
    df_orders
    # Handle empty/null payment modes
    .withColumn(
        "Payment_Mode",
        F.when(
            F.col("Payment_Mode").isNull() |
            (F.length(F.trim(F.col("Payment_Mode"))) == 0),
            "unknown"
        ).otherwise(F.trim(F.col("Payment_Mode")))
    )
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 16, Finished, Available, Finished, False)

In [15]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d595221-a275-4d0e-890d-5e83e6339eda)

In [16]:
from pyspark.sql import functions as F

df_orders = (
    df_orders
    # Replace null promo codes with placeholder
    .withColumn(
        "Promo_Code",
        F.coalesce(
            F.col("Promo_Code"),
            F.lit("NO_PROMO")
        )
    )
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 18, Finished, Available, Finished, False)

In [17]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc983102-2d38-4240-bb55-2559fd20baea)

In [18]:
from pyspark.sql import functions as F

df_orders = (
    df_orders
    .withColumn(
        "Order_Amount",
        F.abs(
            F.regexp_replace(
                F.regexp_replace(
                    F.col("Order_Amount$"),
                    r"(?i)(USD|INR|RS\.?|₹|\$|RUPAY|Rupees)",
                    ""
                ),
                ",",
                ""
            ).cast("double")
        )
    )
    .drop("Order_Amount$")
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 20, Finished, Available, Finished, False)

In [19]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 404336ae-61fc-467c-a247-1956887b67a2)

In [20]:
from pyspark.sql import functions as F

df_orders = (
    df_orders
    .withColumn("Order_Year", F.year(F.col("Order_Date")))
    .withColumn("Order_Month", F.month(F.col("Order_Date")))
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 22, Finished, Available, Finished, False)

In [21]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4e74bb2-2845-4652-83fd-2894a238d4c2)

In [22]:
from pyspark.sql import functions as F

df_orders = (
    df_orders
    .withColumn(
        "Order_Hash",
        F.sha2(
            F.concat_ws("|", *[F.col(c) for c in df_orders.columns]),
            256
        )
    )
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 24, Finished, Available, Finished, False)

In [23]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d3b493d-ac80-4ab8-9df0-6c94d396288a)

In [24]:
df_orders = (
    df_orders

    # Drop rows where critical fields are missing
    .dropna(subset=["Order_ID", "Order_Amount", "CustomerID"])

    # Remove duplicate Order IDs
    .dropDuplicates(["Order_ID"])
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 26, Finished, Available, Finished, False)

In [25]:
display(df_orders)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 852345f1-ba65-458f-bf99-f6a7e77c91d7)

In [26]:
# Save cleaned Orders data
df_orders.write.mode("overwrite").format("delta").saveAsTable("silver_orders")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 28, Finished, Available, Finished, False)

****Cleaning Inventory Data****

In [27]:
from pyspark.sql.functions import *

# Your actual raw DataFrame
df_inventory_cleaned = df_inventory

# Rename raw columns first
df_inventory_cleaned = (
    df_inventory_cleaned
    .withColumnRenamed("product_id", "ProductID")
    .withColumnRenamed("productName", "ProductName")
    .withColumnRenamed("stock", "RawStock")
    .withColumnRenamed("last_stocked", "LastStocked")
    .withColumnRenamed("warehouse", "RawWarehouse")
    .withColumnRenamed("cost_price", "CostPrice")
    .withColumnRenamed("available", "RawAvailable")
)

# Product name
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "ProductName",
    initcap(trim(col("ProductName")))
)

# Stock
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "Stock",
    when(
        regexp_extract(col("RawStock"), r"(\d+)", 1) != "",
        regexp_extract(col("RawStock"), r"(\d+)", 1).cast("int")
    )
    .when(lower(trim(col("RawStock"))) == "twenty five", 25)
    .when(lower(trim(col("RawStock"))) == "twenty", 20)
    .when(lower(trim(col("RawStock"))) == "eighteen", 18)
    .when(lower(trim(col("RawStock"))) == "fifteen", 15)
    .when(lower(trim(col("RawStock"))) == "twelve", 12)
    .otherwise(None)
    .cast("int")
)

# Normalize dates
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "LastStocked",
    coalesce(
        to_date(col("LastStocked"), "yyyy/MM/dd"),
        to_date(col("LastStocked"), "dd-MM-yyyy"),
        to_date(col("LastStocked"), "yyyy.MM.dd"),
        to_date(col("LastStocked"), "dd/MM/yyyy"),
        to_date(col("LastStocked"), "MM-dd-yyyy"),
        to_date(col("LastStocked"), "yyyy-MM-dd")
    )
)

# Cost price
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "CostPrice",
    regexp_extract(col("CostPrice"), r"(\d+(?:\.\d+)?)", 1)
    .cast("double")
)

# Warehouse
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "Warehouse",
    initcap(
        trim(
            regexp_replace(col("RawWarehouse"), r"[^a-zA-Z0-9\s]", " ")
        )
    )
)

# Available
df_inventory_cleaned = df_inventory_cleaned.withColumn(
    "Available",
    when(lower(trim(col("RawAvailable"))).isin("yes", "y", "true"), True)
    .when(lower(trim(col("RawAvailable"))).isin("no", "n", "false"), False)
    .otherwise(None)
)

# Keep only final Silver columns
df_inventory_cleaned = df_inventory_cleaned.select(
    "ProductID",
    "ProductName",
    "Stock",
    "LastStocked",
    "Warehouse",
    "CostPrice",
    "Available"
)

# Remove duplicate products
df_inventory_cleaned = df_inventory_cleaned.dropDuplicates(["ProductID"])

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 29, Finished, Available, Finished, False)

In [29]:
display(df_inventory)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 48b9856f-590a-40da-88fe-2580fd0567dc)

In [30]:
# Optional: Save to Silver Layer
df_inventory_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_inventory")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 32, Finished, Available, Finished, False)

****Cleaning Returns Data****

In [31]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

# ============================================================
# 1. Rename Prop_* columns using the actual Returns structure
# ============================================================

df_returns_cleaned = (
    df_returns
    .withColumnRenamed("Prop_0", "ReturnID")
    .withColumnRenamed("Prop_1", "OrderID")
    .withColumnRenamed("Prop_2", "CustomerID")
    .withColumnRenamed("Prop_3", "Product")
    .withColumnRenamed("Prop_4", "ReturnReason")
    .withColumnRenamed("Prop_5", "ReturnDate")
    .withColumnRenamed("Prop_6", "RefundStatus")
    .withColumnRenamed("Prop_7", "PickupAddress")
    .withColumnRenamed("Prop_8", "ReturnAmount")
)

# ============================================================
# 2. Remove the accidental header row
# ============================================================

df_returns_cleaned = df_returns_cleaned.filter(
    lower(trim(col("ReturnID"))) != "return_id"
)

# ============================================================
# 3. Clean ReturnID
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnID",
    trim(upper(col("ReturnID")))
)

# ============================================================
# 4. Clean OrderID
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "OrderID",
    trim(upper(col("OrderID")))
)

# ============================================================
# 5. Clean CustomerID
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "CustomerID",
    trim(upper(col("CustomerID")))
)

# ============================================================
# 6. Clean Product
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "Product",
    initcap(
        trim(
            regexp_replace(
                col("Product"),
                r"[^a-zA-Z0-9\s]",
                " "
            )
        )
    )
)

# ============================================================
# 7. Clean ReturnReason
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnReason",
    initcap(
        trim(
            regexp_replace(
                col("ReturnReason"),
                r"[^a-zA-Z0-9\s]",
                " "
            )
        )
    )
)

# ============================================================
# 8. Clean ReturnDate
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnDate",
    coalesce(
        to_date(col("ReturnDate"), "yyyy/MM/dd"),
        to_date(col("ReturnDate"), "dd-MM-yyyy"),
        to_date(col("ReturnDate"), "yyyy.MM.dd"),
        to_date(col("ReturnDate"), "dd/MM/yyyy"),
        to_date(col("ReturnDate"), "MM-dd-yyyy"),
        to_date(col("ReturnDate"), "yyyy-MM-dd")
    )
)

# ============================================================
# 9. Clean RefundStatus
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "RefundStatus",
    lower(
        trim(
            regexp_replace(
                col("RefundStatus"),
                r"[^a-zA-Z\s]",
                ""
            )
        )
    )
)

# ============================================================
# 10. Clean PickupAddress
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "PickupAddress",
    initcap(
        trim(
            regexp_replace(
                col("PickupAddress"),
                r"[^a-zA-Z0-9\s]",
                " "
            )
        )
    )
)

# ============================================================
# 11. Clean ReturnAmount
# ============================================================

df_returns_cleaned = df_returns_cleaned.withColumn(
    "ReturnAmount",
    regexp_extract(
        col("ReturnAmount"),
        r"(\d+(?:\.\d+)?)",
        1
    ).cast(DoubleType())
)

# ============================================================
# 12. Remove invalid records
# ============================================================

df_returns_cleaned = df_returns_cleaned.filter(
    col("ReturnID").isNotNull()
)

# ============================================================
# 13. Remove duplicate Return IDs
# ============================================================

df_returns_cleaned = df_returns_cleaned.dropDuplicates(
    ["ReturnID"]
)


StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 33, Finished, Available, Finished, False)

In [32]:
# Display the cleaned data
display(df_returns_cleaned)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f0f613a0-c72b-427e-ab9b-6b7a94cbf6f5)

In [33]:
df_returns_cleaned.write.mode("overwrite").format("delta").saveAsTable("silver_returns")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 35, Finished, Available, Finished, False)

****GOLD LAYER – Aggregation & KPIs****


 **Enrich Orders with Returns & Inventory****

In [34]:
from pyspark.sql.functions import *

# Load Silver Tables
df_orders = spark.read.table("silver_orders")
df_returns = spark.read.table("silver_returns")
df_inventory = spark.read.table("silver_inventory")


StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 36, Finished, Available, Finished, False)

In [35]:
print("Orders:", df_orders.columns)
print("Returns:", df_returns.columns)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 37, Finished, Available, Finished, False)

Orders: ['Order_ID', 'Product_Name', 'Qty', 'Order_Date', 'Delivery_Status', 'Payment_Mode', 'Ship_Address', 'Email', 'Promo_Code', 'Feedback_Score', 'Order_Amount', 'CustomerID', 'Order_Year', 'Order_Month', 'Order_Hash']
Returns: ['ReturnID', 'OrderID', 'CustomerID', 'Product', 'ReturnReason', 'ReturnDate', 'RefundStatus', 'PickupAddress', 'ReturnAmount']


In [36]:
from pyspark.sql import functions as F

df_returns = (
    df_returns
    .withColumnRenamed("Return_ID", "ReturnID")
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Customer_ID", "CustomerID")
    .withColumnRenamed("Return_Reason", "ReturnReason")
    .withColumnRenamed("Return_Date", "ReturnDate")
    .withColumnRenamed("Refund_Status", "RefundStatus")
    .withColumnRenamed("Pickup_Address", "PickupAddress")
    .withColumnRenamed("Return_Amount", "ReturnAmount")
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 38, Finished, Available, Finished, False)

In [38]:
from pyspark.sql import functions as F

df_orders = (
    df_orders
    .withColumnRenamed("Order_ID", "OrderID")
    .withColumnRenamed("Product_Name", "ProductName")
    .withColumnRenamed("Order_Date", "OrderDate")
    .withColumnRenamed("Delivery_Status", "DeliveryStatus")
    .withColumnRenamed("Payment_Mode", "PaymentMode")
    .withColumnRenamed("Ship_Address", "ShipAddress")
    .withColumnRenamed("Promo_Code", "PromoCode")
    .withColumnRenamed("Feedback_Score", "FeedbackScore")
    .withColumnRenamed("Order_Amount", "OrderAmount")
    .withColumnRenamed("Order_Year", "OrderYear")
    .withColumnRenamed("Order_Month", "OrderMonth")
    .withColumnRenamed("Order_Hash", "OrderHash")
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 40, Finished, Available, Finished, False)

In [39]:
# STEP 1: Join Orders with Returns
# LEFT JOIN to retain all orders

df_order_return = df_orders.join(
    df_returns,
    on="OrderID",
    how="left"
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 41, Finished, Available, Finished, False)

In [40]:
display(df_order_return)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6443960c-b22f-4a7b-99dd-2e719c8369b0)

In [41]:
#STEP 2: Join Inventory (LEFT JOIN on cleaned Product_Name)
df_enriched = df_order_return.join(
    df_inventory,
    df_order_return.ProductName == df_inventory.ProductName,
    "left"
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 43, Finished, Available, Finished, False)

In [42]:
display(df_enriched)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f870c26e-8f65-401b-bfac-90a2d4b799c6)

In [46]:
# 3. Aggregate Returns by OrderID
# ============================================================

df_return_summary = (
    df_returns
    .groupBy("OrderID")
    .agg(
        countDistinct("ReturnID").alias("Return_Count"),
        first("ReturnDate", ignorenulls=True).alias("ReturnDate"),
        first("RefundStatus", ignorenulls=True).alias("RefundStatus"),
        round(sum("ReturnAmount"), 2).alias("Total_Return_Amount")
    )
)



StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 48, Finished, Available, Finished, False)

In [47]:
# 4. Join Orders + Returns
# ============================================================

df_order_return = (
    df_orders
    .join(
        df_return_summary,
        on="OrderID",
        how="left"
    )
)


StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 49, Finished, Available, Finished, False)

In [48]:
# 5. Replace NULL return values
# ============================================================

df_order_return = (
    df_order_return
    .withColumn(
        "Return_Count",
        coalesce(col("Return_Count"), lit(0))
    )
    .withColumn(
        "Total_Return_Amount",
        coalesce(col("Total_Return_Amount"), lit(0.0))
    )
)


StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 50, Finished, Available, Finished, False)

In [49]:
# 6. Join Inventory
# ============================================================

df_enriched = (
    df_order_return
    .join(
        df_inventory,
        df_order_return.ProductName == df_inventory.ProductName,
        "left"
    )
    .drop(df_inventory.ProductName)
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 51, Finished, Available, Finished, False)

In [50]:
# 7. Calculate COGS
# ============================================================

df_enriched = df_enriched.withColumn(
    "COGS",
    col("Qty") * col("CostPrice")
)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 52, Finished, Available, Finished, False)

In [51]:
# 8. Calculate Net Profit
# ============================================================

df_enriched = df_enriched.withColumn(
    "Net_Profit",
    col("OrderAmount") - col("COGS")
)


StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 53, Finished, Available, Finished, False)

In [52]:
# 9. Product-level KPIs
# ============================================================

df_kpi = (
    df_enriched
    .groupBy("ProductName")
    .agg(
        countDistinct("OrderID").alias("Total_Orders"),

        countDistinct("CustomerID").alias("Unique_Customers"),

        sum("Qty").alias("Total_Quantity_Sold"),

        round(sum("OrderAmount"), 2).alias("Total_Revenue"),

        round(avg("OrderAmount"), 2).alias("Avg_Order_Value"),

        round(sum("COGS"), 2).alias("Total_COGS"),

        round(avg("CostPrice"), 2).alias("Avg_Cost"),

        round(sum("Net_Profit"), 2).alias("Net_Profit"),

        sum("Return_Count").alias("Total_Returns"),

        countDistinct(
            when(
                col("Return_Count") > 0,
                col("OrderID")
            )
        ).alias("Returned_Orders"),

        round(
            sum("Total_Return_Amount"),
            2
        ).alias("Total_Return_Amount"),

        max("Stock").alias("Current_Stock")
    )
)



StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 54, Finished, Available, Finished, False)

In [53]:
# 10. Return Rate
# ============================================================

df_kpi = df_kpi.withColumn(
    "Return_Rate_Percent",
    round(
        (col("Returned_Orders") / col("Total_Orders")) * 100,
        2
    )
)


StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 55, Finished, Available, Finished, False)

In [54]:
# 11. Save Gold table
# ============================================================

df_kpi.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("gold_product_kpis")



StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 56, Finished, Available, Finished, False)

In [55]:
display(df_kpi)

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6b1bb018-358f-411a-a72a-763b60ec45de)

In [56]:
from notebookutils import mssparkutils

# Bronze
mssparkutils.fs.mkdirs("Files/Bronze/Orders")
mssparkutils.fs.mkdirs("Files/Bronze/Returns")
mssparkutils.fs.mkdirs("Files/Bronze/Inventory")

# Silver
mssparkutils.fs.mkdirs("Files/Silver/Orders")
mssparkutils.fs.mkdirs("Files/Silver/Returns")
mssparkutils.fs.mkdirs("Files/Silver/Inventory")

# Gold
mssparkutils.fs.mkdirs("Files/Gold/Product_KPIs")

print("Folder structure created successfully!")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 58, Finished, Available, Finished, False)

Folder structure created successfully!


In [59]:
df_orders.write.mode("overwrite").format("delta").save("Files/Bronze/Orders")

df_returns.write.mode("overwrite").format("delta").save("Files/Bronze/Returns")

df_inventory.write.mode("overwrite").format("delta").save("Files/Bronze/Inventory")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 61, Finished, Available, Finished, False)

In [60]:
df_orders.write.mode("overwrite").format("delta").save("Files/Silver/Orders")

df_returns_cleaned.write.mode("overwrite").format("delta").save("Files/Silver/Returns")

df_inventory_cleaned.write.mode("overwrite").format("delta").save("Files/Silver/Inventory")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 62, Finished, Available, Finished, False)

In [61]:
df_kpi.write \
    .mode("overwrite") \
    .format("delta") \
    .save("Files/Gold/Product_KPIs")

StatementMeta(, c56e6910-6434-43c4-b76d-091d1b26ec97, 63, Finished, Available, Finished, False)